# InduSense — Sprint 1 : cartes d'identité des fichiers

Pour chaque fichier de `datas/` (CSV et script SQL), ce notebook produit une carte d'identité Markdown dans
`artifacts/ingestion/identities/<AAAAMMJJ>/<nom_fichier>_<HHMMSS>.md` : source, taille, horaires de traitement,
charset, dimensions, types de colonnes inférés et statistiques des colonnes numériques et dates.

Le code est dans le paquet `src/indusense/` : chaque section ci-dessous explique un module et renvoie vers lui.
Le même code sert la ligne de commande, qui traite un fichier à la fois sans ouvrir le notebook :

```shell
uv run indusense datas/telemetry.csv
```

In [1]:
from pathlib import Path

from IPython.display import Markdown, display

from indusense import describer, write_card

DATA_DIR = Path("datas")
CSV_FILES = [DATA_DIR / "releves_incidents.csv", DATA_DIR / "telemetry.csv"]
SQL_FILE = DATA_DIR / "machine.sql.txt"

## 1. Lecture du fichier : taille, charset et séparateur

- **Charset** : on essaie de décoder le fichier en ASCII, puis en UTF-8 (BOM éventuel ignoré), puis en
  Windows-1252 : le Latin-1 de Windows, qui code en plus `œ`, `’`, `€`... (fréquents en français), là où
  Latin-1 les lirait comme des caractères de contrôle. À défaut, le fichier est lu en ISO-8859-1 (Latin-1),
  qui accepte n'importe quel octet.
- **Séparateur** : il n'est pas toujours une virgule, `csv.Sniffer` le détecte parmi `,` `;` `\t` `|`.

Code : [`src/indusense/reading.py`](src/indusense/reading.py)

## 2. Inférence du type des colonnes

| Type | Règle |
| --- | --- |
| `IDENTITY` | nom `id` / `*_id` / `*_code`, ou valeurs de la forme `MACH-01`, `INC-000001`, `OP1002` (au moins 2 chiffres), ou codes en chiffres à zéro initial (`0012`) |
| `BOOLEAN` | valeurs uniquement `0` / `1` / `true` / `false` (casse ignorée, mélange possible) |
| `INTEGER` / `FLOAT` | colonne numérique ; décimales avec point (`12.5`) ou virgule (`12,5`), éventuellement mélangées ; une colonne sans décimale reste `INTEGER` même avec des valeurs manquantes |
| `TIME` | heures `HH:MM`, `HH:MM:SS` ou `HH:MM:SS.fff` (millisecondes ; `5:42` accepté) |
| `DATE` / `DATETIME` | ISO 8601, `JJ/MM/AAAA`, `MM/JJ/AAAA`, `JJ-MM-AAAA` ou `AAAA/MM/JJ`, avec ou sans heure ; `DATETIME` si les valeurs portent une heure |
| `CATEGORY` | texte avec au plus max(20, 1 % des lignes) valeurs distinctes |
| `TEXT` | autre texte |

**Lecture des valeurs.** Le CSV est lu entièrement en texte : pandas ne convertit rien, toute l'inférence est
faite ici. Ainsi un code `0012` n'est pas transformé en nombre `12` : un zéro initial signale un code, la
colonne est typée `IDENTITY`. Les espaces autour des valeurs sont retirés (`nuit ` et `nuit` sont la même catégorie).
Seules les valeurs vides et `null` (casse ignorée) sont manquantes et ignorées. Par défaut, pandas reconnaît
aussi `NA`, `None`, `n/a`... : ces valeurs, qui peuvent être de vraies données, sont désormais gardées. Dans
le script SQL, `NULL` est déjà lu comme manquant par sqlglot.

**Tolérance.** Une colonne numérique, heure ou date est retenue si **au moins 95 %** des valeurs sont lisibles :
une faute de saisie isolée ne la fait plus basculer en `TEXT` ou `CATEGORY`. Pour les heures et dates, les
formats d'une même famille sont combinés : une colonne peut mêler `HH:MM` et `HH:MM:SS`, ou `JJ/MM/AAAA` avec
et sans heure. On garde la famille qui lit le plus de valeurs. Les remarques de lecture sont ajoutées au type
dans la carte :

- `2 invalid values` : valeurs illisibles (colonne numérique, heure ou date), exclues des statistiques,
- `ambiguous day/month, read as DD/MM` : tous les jours sont ≤ 12, l'ordre jour/mois ne peut pas être déduit
  (le jour avant le mois est retenu par défaut),
- `UTC` : les valeurs portent un fuseau horaire, elles sont ramenées en UTC (min / max suffixés de `UTC`).

Code : [`src/indusense/inference.py`](src/indusense/inference.py)

## 3. Statistiques des colonnes numériques et dates

Min et max pour toutes ces colonnes ; mean et median uniquement pour les numériques.

Code : [`src/indusense/stats.py`](src/indusense/stats.py)

## 4. Construction de la carte d'identité

`write_card` produit l'en-tête commun à tous les fichiers (source, taille, horaires, charset) puis délègue
le corps de la carte à une fonction `describe` propre au format : `describe_csv` ici, `describe_sql` en section 5.

Code : [`src/indusense/card.py`](src/indusense/card.py)

## 5. Pour aller plus loin : carte d'identité du script SQL

`machine.sql.txt` est un script PostgreSQL : il ne se lit pas comme un tableau. Il faut un parseur SQL
capable d'en extraire deux informations :

- `CREATE TABLE` donne les colonnes et leur **type SQL déclaré**,
- `INSERT INTO ... VALUES` donne les **données**, chargées dans un DataFrame par table.

Chaque table est ensuite décrite comme un CSV, avec la même fonction `describe_frame`. Les colonnes déclarées
mais absentes des `INSERT` (remplies par une valeur par défaut, ex. `created_at`) ont le type inféré `-`.

### Choix de la librairie : sqlglot

| Option | Ce qu'elle fournit | Pourquoi elle n'a pas été retenue |
| --- | --- | --- |
| **[sqlglot](https://github.com/tobymao/sqlglot)** ✅ | Arbre syntaxique complet (AST) : tables, colonnes, types, valeurs déjà typées (`int`, `Decimal`, `None`...) | — |
| [sqlparse](https://github.com/andialbrecht/sqlparse) | Découpage en instructions et en *tokens* (parseur « non validant ») | Pas d'arbre syntaxique : il faudrait reconstruire à la main colonnes, types et lignes de `VALUES` en parcourant les tokens |
| [pglast](https://github.com/lelit/pglast) | Le vrai parseur de PostgreSQL (libpg_query) | Extension C : demande des binaires compilés pour notre version de Python (3.15) ; limitée au seul dialecte PostgreSQL |
| `sqlite3` (bibliothèque standard) | Exécute le script dans une base en mémoire, puis on relit les tables | Ce n'est pas un parseur : il faut que SQLite accepte le script, or celui-ci utilise des syntaxes propres à PostgreSQL (`DEFAULT NOW()`, `TIMESTAMPTZ`...) ; et on perd les types déclarés |

sqlglot coche toutes les cases pour ce besoin :

- **Python pur, sans aucune dépendance** : `uv add sqlglot` n'installe qu'un seul paquet, rien à compiler,
- **comprend le dialecte PostgreSQL** (`read="postgres"`) : `TIMESTAMPTZ`, `NUMERIC(6,2)`, `ON CONFLICT ... DO UPDATE`...,
- **aucune base de données nécessaire** : le script est analysé, jamais exécuté,
- **arbre syntaxique navigable** : `exp.Create`, `exp.Insert`, `exp.ColumnDef`... rendent l'extraction directe,
- projet très utilisé et activement maintenu.

```shell
uv add sqlglot
```

Code : [`src/indusense/sql.py`](src/indusense/sql.py)

## 6. Génération des cartes d'identité

`describer` choisit la fonction `describe` selon le fichier : `describe_sql` pour un script SQL (`.sql`,
`.sql.txt`...), `describe_csv` pour un CSV (`.csv`, `.tsv`) ;
tout autre fichier est refusé. La ligne de commande (`src/indusense/cli.py`) fait la même chose
pour les fichiers qu'on lui passe.

In [2]:
for path in [*CSV_FILES, SQL_FILE]:
    output = write_card(path, describer(path))
    print(f"Carte générée : {output}")
    display(Markdown(output.read_text(encoding="utf-8")))

Carte générée : artifacts/ingestion/identities/20261120/releves_incidents_160451.md


# Carte d'identité : releves_incidents.csv

- Source: /home/thopuetai/Formation IA/indusense/datas/releves_incidents.csv
- Filesize: 134.64 Ko
- ProcessBeginAt: 16:04:51
- ProcessEndAt: 16:04:52
- Charset: UTF-8
- Rows Number: 1245
- Cols Number: 18

## Column list

| Colname | Inferred Type |
| --- | --- |
| incident_id | IDENTITY |
| date | DATE |
| time | TIME |
| operator_name | CATEGORY |
| machine_id | IDENTITY |
| severity | INTEGER |
| operator_badge | IDENTITY |
| comment | TEXT |
| shift | CATEGORY |
| type_surchauffe | BOOLEAN |
| type_baisse_pression | BOOLEAN |
| type_vibration | BOOLEAN |
| type_bruit_mecanique | BOOLEAN |
| type_surconsommation | BOOLEAN |
| type_blocage_mecanique | BOOLEAN |
| type_alarme_capteur | BOOLEAN |
| type_arret_urgence | BOOLEAN |
| type_defaut_qualite | BOOLEAN |

## Numeric and date columns

| Colname | Min value | Max value | Mean value | Median value |
| --- | --- | --- | --- | --- |
| date | 2025-06-01 | 2026-06-08 | - | - |
| time | 00:06 | 23:55 | - | - |
| severity | 2 | 5 | 2.925 | 3.000 |


Carte générée : artifacts/ingestion/identities/20261120/telemetry_160452.md


# Carte d'identité : telemetry.csv

- Source: /home/thopuetai/Formation IA/indusense/datas/telemetry.csv
- Filesize: 7.94 Mo
- ProcessBeginAt: 16:04:52
- ProcessEndAt: 16:04:53
- Charset: ASCII
- Rows Number: 135626
- Cols Number: 7

## Column list

| Colname | Inferred Type |
| --- | --- |
| machine_id | IDENTITY |
| timestamp | DATETIME |
| temperature_c | FLOAT |
| pressure_bar | FLOAT |
| voltage_mean_v | FLOAT |
| rotation_mean_rpm | FLOAT |
| pieces_produced | INTEGER |

## Numeric and date columns

| Colname | Min value | Max value | Mean value | Median value |
| --- | --- | --- | --- | --- |
| timestamp | 2025-06-01 00:00:00 | 2026-06-08 23:00:00 | - | - |
| temperature_c | 32.000 | 80.000 | 48.183 | 48.055 |
| pressure_bar | 159.982 | 215.814 | 199.770 | 199.866 |
| voltage_mean_v | 220.900 | 242.000 | 227.631 | 227.420 |
| rotation_mean_rpm | 1100.000 | 1900.000 | 1589.205 | 1590.370 |
| pieces_produced | 0 | 114 | 49.533 | 49.000 |


Carte générée : artifacts/ingestion/identities/20261120/machine_160453.md


# Carte d'identité : machine.sql.txt

- Source: /home/thopuetai/Formation IA/indusense/datas/machine.sql.txt
- Filesize: 22.41 Ko
- ProcessBeginAt: 16:04:53
- ProcessEndAt: 16:04:53
- Charset: UTF-8
- SQL Dialect: postgres
- Statements Number: 10
- Tables Number: 2

## Table `machine`

- Rows Number: 15
- Cols Number: 11

### Column list

| Colname | SQL Type | Inferred Type |
| --- | --- | --- |
| machine_code | VARCHAR(16) | IDENTITY |
| commissioning_date | DATE | DATE |
| max_daily_capacity | INT | INTEGER |
| max_hourly_capacity_pieces | INT | INTEGER |
| model | VARCHAR(32) | CATEGORY |
| production_line | VARCHAR(16) | CATEGORY |
| location | VARCHAR(16) | CATEGORY |
| criticality | VARCHAR(8) | CATEGORY |
| is_active | BOOLEAN | - |
| created_at | TIMESTAMPTZ | - |
| updated_at | TIMESTAMPTZ | - |

### Numeric and date columns

| Colname | Min value | Max value | Mean value | Median value |
| --- | --- | --- | --- | --- |
| commissioning_date | 2019-07-23 | 2025-05-25 | - | - |
| max_daily_capacity | 750 | 1428 | 1054.867 | 1027.000 |
| max_hourly_capacity_pieces | 47 | 89 | 65.867 | 64.000 |

## Table `maintenance`

- Rows Number: 115
- Cols Number: 11

### Column list

| Colname | SQL Type | Inferred Type |
| --- | --- | --- |
| maintenance_id | INT | IDENTITY |
| machine_code | VARCHAR(16) | IDENTITY |
| maintenance_at | TIMESTAMPTZ | DATETIME (UTC) |
| maintenance_type | VARCHAR(16) | CATEGORY |
| action_type | VARCHAR(32) | CATEGORY |
| component | VARCHAR(64) | CATEGORY |
| description | TEXT | TEXT |
| related_incident_id | VARCHAR(16) | IDENTITY |
| duration_hours | DECIMAL(6, 2) | FLOAT |
| created_at | TIMESTAMPTZ | - |
| updated_at | TIMESTAMPTZ | - |

### Numeric and date columns

| Colname | Min value | Max value | Mean value | Median value |
| --- | --- | --- | --- | --- |
| maintenance_at | 2025-06-15 06:00:00 UTC | 2026-05-30 22:00:00 UTC | - | - |
| duration_hours | 1.000 | 8.000 | 2.943 | 2.640 |
